# ReviewLens: a GenAI Customer Feedback Analyser
**Next Token Weekly · Issue 01 · Industry project**

**Business problem.** An online electronics retailer receives ~10,000 product reviews a month. The head of customer experience wants to know every week:
1. What are customers unhappy about?
2. How urgent is it?
3. Is it getting better or worse?

**What you will build** (every step maps to a chapter of Issue 01):

| Step | What | Concept |
|---|---|---|
| 1 | Load and clean reviews | pandas |
| 2 | Token audit and cost projection | Ch 4 Tokens |
| 3 | Sentence embeddings + semantic search | Ch 5 Embeddings |
| 4 | Theme discovery with K-means + PCA map | Ch 5 |
| 5 | Structured LLM labelling at temperature 0 | Ch 3, 7, 8 |
| 6 | Validation and evaluation against gold labels | Ch 9 Hallucination |
| 7 | Insight dashboard | Analytics |
| 8 | Executive memo | GenAI summarisation |

> **Which LLM?** The notebook uses **OpenAI** if your key works, otherwise the **Google Gemini free tier** (free key from aistudio.google.com), otherwise a simple rule-based labeller, so it always runs end to end. Compare the LLM labels with the rules to see what GenAI adds.

## 0 · Setup

In [ ]:
# Run once. Remove the leading '#' to install.
# %pip install -q pandas numpy matplotlib scikit-learn tiktoken sentence-transformers openai

In [ ]:
import os, re, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42); np.random.seed(42)
pd.set_option("display.max_colwidth", 120)

### Choose the LLM provider: OpenAI → Google Gemini (free) → rules
The notebook tries providers **in order** and uses the first one that actually answers:
1. **OpenAI**, if `OPENAI_API_KEY` is set and the key works (not out of credit).
2. **Google Gemini free tier**, if `GEMINI_API_KEY` is set. Get a free key at **aistudio.google.com → Get API key** (Google account needed, no card).
3. **Rule-based labeller**, so the notebook always runs.

Copy **`.env.example`** (repo root) to **`.env`** and add your key. `.env` is listed in `.gitignore`, so your key never gets pushed to GitHub. Never paste keys into the notebook itself.
```
GEMINI_API_KEY=your-gemini-key
# OPENAI_API_KEY=your-openai-key
```
Gemini is called through Google's **OpenAI-compatible endpoint**, so the same `openai` Python client works for both providers.

In [ ]:
def load_dotenv(path=".env"):
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
load_dotenv(".env"); load_dotenv("../.env")   # week folder first, then the repo root

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
CANDIDATES = []
if os.getenv("OPENAI_API_KEY"):
    CANDIDATES.append(dict(provider="openai", key=os.getenv("OPENAI_API_KEY"),
                           base_url=os.getenv("LLM_BASE_URL"),          # None = api.openai.com
                           model=os.getenv("LLM_MODEL", "gpt-4o-mini"),
                           batch=int(os.getenv("BATCH_SIZE", 10)), rpm=float(os.getenv("OPENAI_RPM", 300))))
gkey = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if gkey:
    CANDIDATES.append(dict(provider="gemini", key=gkey, base_url=GEMINI_BASE_URL,
                           model=os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
                           batch=int(os.getenv("BATCH_SIZE", 20)),           # fewer calls = stays inside free quota
                           rpm=float(os.getenv("GEMINI_RPM", 8))))            # free tier allows only a few requests/minute

PROVIDER, client, LLM_MODEL, BATCH_SIZE, MIN_GAP = "rules", None, None, 1, 0.0
for c in CANDIDATES:
    try:
        from openai import OpenAI
        cl = OpenAI(api_key=c["key"], base_url=c["base_url"]) if c["base_url"] else OpenAI(api_key=c["key"])
        cl.chat.completions.create(model=c["model"], messages=[{"role": "user", "content": "Reply with OK"}], max_tokens=5)
        PROVIDER, client, LLM_MODEL, BATCH_SIZE, MIN_GAP = c["provider"], cl, c["model"], c["batch"], 60 / c["rpm"]
        break
    except Exception as e:
        print(f"{c['provider']} not available -> {type(e).__name__}: {str(e)[:150]}")

USE_LLM = PROVIDER != "rules"
print(f"Provider: {PROVIDER.upper()} | model: {LLM_MODEL or 'rule-based fallback'} | batch size: {BATCH_SIZE}")

_last_call = [0.0]
def throttle():
    # Wait so we never exceed the provider's requests-per-minute limit.
    wait = MIN_GAP - (time.time() - _last_call[0])
    if wait > 0: time.sleep(wait)
    _last_call[0] = time.time()

## 1 · Load and clean the data
We start with **120 sample reviews with known (gold) labels** in `data/sample_reviews.csv`, so you can measure accuracy right away. (If the file is missing, this cell regenerates it.)
To use real data, put your CSV in `data/` and set `DATA_PATH` to it; keep a column called `review` (and `date` if you want the weekly trend).
Good sources: Amazon Reviews 2023 (McAuley Lab, Hugging Face), `amazon_polarity` (Hugging Face), Flipkart product reviews (Kaggle), Yelp Open Dataset.

In [ ]:
TEMPLATES = {
 ("delivery","negative"): ["My order arrived {n} days late and the box was crushed.",
                           "Delivery was promised for Monday, it came the next week. Very disappointed.",
                           "Package never arrived and tracking stopped updating.",
                           "Courier left it at the wrong address, took ages to sort out."],
 ("delivery","positive"): ["Arrived a day early, well packed. Great delivery!",
                           "Super fast shipping, got it within {n} days."],
 ("quality","negative"):  ["Screen cracked after {n} weeks of normal use.",
                           "Cheap plastic, the hinge broke in a month.",
                           "Stopped working after {n} days, feels poorly made."],
 ("quality","positive"):  ["Build quality is excellent, feels premium.",
                           "Solid product, still working perfectly after {n} months."],
 ("battery","negative"):  ["Battery drains from 100 to 20 percent in {n} hours.",
                           "Battery life is terrible, I charge it twice a day.",
                           "Phone gets hot and the battery dies quickly."],
 ("battery","positive"):  ["Battery easily lasts two full days, impressive."],
 ("price","negative"):    ["Way too expensive for what you get.",
                           "Price dropped by {n}00 rupees a week after I bought it. Feel cheated."],
 ("price","positive"):    ["Great value for money at this price.",
                           "Best budget option, worth every rupee."],
 ("support","negative"):  ["Customer care kept me on hold for {n} minutes and then hung up.",
                           "Raised a warranty claim, no response for {n} days.",
                           "Support agent was rude and did not solve my issue."],
 ("support","positive"):  ["Support replaced my unit within {n} days, very helpful team."],
 ("usability","negative"):["The app is confusing and setup took forever.",
                           "Too many settings, very hard to use for my parents."],
 ("usability","positive"):["Very easy to set up, worked out of the box.",
                           "Simple and intuitive, my kids figured it out in minutes."],
 ("quality","neutral"):   ["It is okay. Does the job, nothing special."],
 ("price","neutral"):     ["Average product for an average price."],
}
DATA_PATH = "data/sample_reviews.csv"
rows = []
keys = list(TEMPLATES)
weights = np.array([5 if k[1]=="negative" else 2 for k in keys], float); weights /= weights.sum()
start = pd.Timestamp("2026-09-07")
for i in range(120):
    theme, sent = keys[np.random.choice(len(keys), p=weights)]
    text = random.choice(TEMPLATES[(theme, sent)]).format(n=random.randint(2, 9))
    urgency = {"negative": random.choice([3,4,5]), "neutral": 2, "positive": 1}[sent]
    if theme in ("support","delivery") and sent=="negative": urgency = max(urgency, 4)
    rows.append({"review_id": i+1, "date": start + pd.Timedelta(days=int(np.random.randint(0, 77))),
                 "review": text, "gold_sentiment": sent, "gold_theme": theme, "gold_urgency": urgency})
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    print("Loaded", DATA_PATH)
else:
    df = pd.DataFrame(rows)
    os.makedirs("data", exist_ok=True); df.to_csv(DATA_PATH, index=False)
    print("Generated sample data and saved it to", DATA_PATH)

# basic cleaning
df["review"] = df["review"].str.strip().str.replace(r"\s+", " ", regex=True)
df = df[df["review"].str.len() > 0].reset_index(drop=True)
print(df.shape); df.head()

In [ ]:
df["gold_sentiment"].value_counts().plot(kind="bar", title="Gold sentiment (sample data)", color=["#AE2F45","#0A7570","#B86E00"])
plt.ylabel("reviews"); plt.show()

## 2 · Token audit and cost projection (Issue 01, Ch 4)
Models read **tokens**, not words, and APIs bill per token. Before spending money, measure.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")       # tokenizer used by GPT-4o-family models
    count_tokens = lambda t: len(enc.encode(t))
    print("Using tiktoken o200k_base")
except Exception as e:
    count_tokens = lambda t: max(1, round(len(t) / 4))  # rule of thumb: ~4 characters per token
    print("tiktoken not installed, using the 4-characters-per-token estimate")

df["n_words"]  = df["review"].str.split().str.len()
df["n_tokens"] = df["review"].apply(count_tokens)
print(df[["n_words","n_tokens"]].describe().round(1))
print("Tokens per word:", round(df.n_tokens.sum() / df.n_words.sum(), 2))

In [ ]:
# See how a sentence is split (try a Hindi sentence too!)
sample = "Battery drains quickly. बैटरी जल्दी खत्म हो जाती है।"
try:
    print([enc.decode([t]) for t in enc.encode(sample)])
except NameError:
    print("Install tiktoken to see real token splits.")
print("tokens:", count_tokens(sample))

In [ ]:
# Projected monthly cost. CHECK CURRENT PRICES on the provider's pricing page; these are placeholders.
MODELS = {  # name: (USD per 1M input tokens, USD per 1M output tokens)
    "small model":  (0.15, 0.60),
    "large model":  (2.50, 10.00),
}
PROMPT_OVERHEAD, OUTPUT_TOKENS, MONTHLY = 350, 60, 10_000
avg_in = df.n_tokens.mean() + PROMPT_OVERHEAD
cost_table = pd.DataFrame([
    {"model": m, "avg_input_tokens": round(avg_in), "output_tokens": OUTPUT_TOKENS,
     "monthly_cost_usd": round(MONTHLY * (avg_in*pi + OUTPUT_TOKENS*po) / 1e6, 2)}
    for m, (pi, po) in MODELS.items()])
cost_table

## 3 · Embeddings and semantic search (Issue 01, Ch 5)
Each review becomes a vector of 384 numbers. Similar meaning means similar direction, measured by cosine similarity.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")
    embed = lambda texts: emb_model.encode(list(texts), normalize_embeddings=True, show_progress_bar=False)
    EMB = "all-MiniLM-L6-v2 (semantic)"
except Exception as e:
    # Fallback: TF-IDF vectors (keyword-based, not truly semantic) so the notebook still runs
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    _tfidf = TfidfVectorizer(stop_words="english").fit(df["review"])
    embed = lambda texts: normalize(_tfidf.transform(list(texts))).toarray()
    EMB = "TF-IDF fallback (install sentence-transformers for real embeddings)"

X = embed(df["review"])
print("Embedding method:", EMB, "| shape:", X.shape)

In [ ]:
def search(query, k=5):
    q = embed([query])[0]
    sims = X @ q                       # cosine similarity, because vectors are normalised
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({"review": df.loc[top, "review"].values, "cosine": sims[top].round(3)})

# Notice: matches by MEANING, even without shared words (with real embeddings)
search("my parcel is missing")

## 4 · Discover themes with K-means and a 2D map
We let the data suggest themes **before** asking an LLM anything. This is unsupervised learning on embeddings.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

scores = {}
for k in range(3, 10):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    scores[k] = silhouette_score(X, labels)
best_k = max(scores, key=scores.get)
print("Silhouette by k:", {k: round(v,3) for k,v in scores.items()}, "-> best k =", best_k)

df["cluster"] = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(X)
P = PCA(n_components=2, random_state=42).fit_transform(X)
df["pc1"], df["pc2"] = P[:,0], P[:,1]

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
for c, g in df.groupby("cluster"):
    ax.scatter(g.pc1, g.pc2, s=40, alpha=.75, label=f"cluster {c}")
ax.set_title("Review themes: embeddings projected to 2D with PCA"); ax.legend(fontsize=8)
plt.show()

# Read 3 examples per cluster and NAME each cluster yourself (analyst judgement!)
for c, g in df.groupby("cluster"):
    print(f"\n--- cluster {c} ({len(g)} reviews) ---")
    for t in g["review"].head(3): print("  •", t)

## 5 · Structured LLM labelling at temperature 0 (Issue 01, Ch 3, 7, 8)
We ask the model for **strict JSON** so the output is machine-readable, and use **temperature 0** so results are repeatable.

In [ ]:
THEMES = ["delivery","quality","battery","price","support","usability"]
SYSTEM_PROMPT = f"""You label customer reviews for an electronics retailer.
Reply with JSON only. For each review give:
  "sentiment": one of ["positive","neutral","negative"]
  "theme":     one of {THEMES}
  "urgency":   integer 1-5 (5 = customer at risk of churning or legal/safety issue)
  "summary":   at most 12 words
If unsure, pick the closest theme. Never invent facts that are not in the review."""

def parse_json(text):
    # Models sometimes wrap JSON in ```json fences; keep only the outermost {...}.
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    return json.loads(text[text.find("{"): text.rfind("}") + 1])

def label_batch_with_llm(reviews):
    """Label several reviews in ONE call (saves free-tier quota). Returns a list aligned with `reviews`."""
    items = [{"id": i, "review": r} for i, r in enumerate(reviews)]
    user = ('Label every review below. Return ONLY JSON of the form '
            '{"labels": [{"id": 0, "sentiment": "...", "theme": "...", "urgency": 3, "summary": "..."}, ...]} '
            'with exactly one entry per id.\n' + json.dumps(items, ensure_ascii=False))
    use_json_mode = True
    for attempt in range(4):
        try:
            throttle()
            kwargs = dict(model=LLM_MODEL, temperature=0,
                          messages=[{"role": "system", "content": SYSTEM_PROMPT},
                                    {"role": "user", "content": user}])
            if use_json_mode: kwargs["response_format"] = {"type": "json_object"}
            r = client.chat.completions.create(**kwargs)
            by_id = {int(d["id"]): d for d in parse_json(r.choices[0].message.content).get("labels", [])}
            return [by_id.get(i, {}) for i in range(len(reviews))]
        except Exception as e:
            msg = str(e)
            print(f"  attempt {attempt+1} failed: {type(e).__name__}: {msg[:120]}")
            if "response_format" in msg: use_json_mode = False      # provider without JSON mode: rely on the prompt
            time.sleep(min(60, 5 * 2 ** attempt))                     # back off (429 = rate limit on free tier)
    return [{} for _ in reviews]

KEYWORDS = {"delivery":["deliver","arriv","courier","package","shipping","tracking"],
            "battery":["battery","charge","drain","hot"],
            "support":["support","customer care","warranty","agent","hold"],
            "price":["price","expensive","value","budget","rupee","cheated"],
            "usability":["setup","set up","app","easy","intuitive","settings","use"],
            "quality":["quality","broke","crack","plastic","build","working","made"]}
NEG = ["late","never","broke","crack","terrible","rude","drain","expensive","cheated","confusing","hard","no response","disappoint","dies","hot","stopped","wrong"]
POS = ["great","excellent","early","fast","impressive","helpful","easy","value","worth","perfect","premium","solid","intuitive","best"]

def label_with_rules(review):
    t = review.lower()
    theme = max(KEYWORDS, key=lambda k: sum(w in t for w in KEYWORDS[k]))
    n, p = sum(w in t for w in NEG), sum(w in t for w in POS)
    sent = "negative" if n > p else "positive" if p > n else "neutral"
    urg = 1 if sent=="positive" else 2 if sent=="neutral" else (4 if theme in ("support","delivery") else 3)
    return {"sentiment": sent, "theme": theme, "urgency": urg, "summary": review[:60]}

t0 = time.time()
if USE_LLM:
    reviews = df["review"].tolist(); raw = []
    for s in range(0, len(reviews), BATCH_SIZE):
        raw += label_batch_with_llm(reviews[s:s + BATCH_SIZE])
        print(f"  labelled {min(s + BATCH_SIZE, len(reviews))}/{len(reviews)}")
    # Any review the LLM skipped or garbled falls back to the rules, and we record which method was used
    method = ["llm" if d else "rules" for d in raw]
    raw = [d if d else label_with_rules(r) for d, r in zip(raw, reviews)]
else:
    raw, method = [label_with_rules(r) for r in df["review"]], ["rules"] * len(df)
df["label_method"] = method
print(f"Labelled {len(raw)} reviews with {PROVIDER.upper()} in {time.time() - t0:.0f}s |",
      pd.Series(method).value_counts().to_dict())
raw[:3]

## 6 · Validate and evaluate (Issue 01, Ch 9)
Never trust model output blindly. **Validate the schema**, then **measure accuracy against gold labels**, then **read the mistakes**.

In [ ]:
def validate(d):
    if isinstance(d, dict):
        d = {k: (str(v).lower().strip() if k in ("sentiment", "theme") and v is not None else v) for k, v in d.items()}
        try: d["urgency"] = int(d.get("urgency"))
        except (TypeError, ValueError): pass
    ok = (isinstance(d, dict) and d.get("sentiment") in ("positive","neutral","negative")
          and d.get("theme") in THEMES and isinstance(d.get("urgency"), (int, float)) and 1 <= d["urgency"] <= 5)
    return d if ok else {"sentiment": None, "theme": None, "urgency": None, "summary": None}

lab = pd.DataFrame([validate(d) for d in raw]).add_prefix("pred_")
df = pd.concat([df.drop(columns=[c for c in df if c.startswith("pred_")], errors="ignore"), lab], axis=1)
print("Invalid / unparseable outputs:", df.pred_sentiment.isna().sum())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
v = df.dropna(subset=["pred_sentiment"])
print("Sentiment accuracy:", round(accuracy_score(v.gold_sentiment, v.pred_sentiment), 3))
print("Theme accuracy:    ", round(accuracy_score(v.gold_theme, v.pred_theme), 3))
print("Urgency within ±1: ", round((abs(v.gold_urgency - v.pred_urgency) <= 1).mean(), 3))
print(classification_report(v.gold_theme, v.pred_theme, zero_division=0))
ConfusionMatrixDisplay.from_predictions(v.gold_theme, v.pred_theme, xticks_rotation=45, cmap="Blues")
plt.title("Theme: gold vs predicted"); plt.show()

In [ ]:
# Error analysis: read the misses. Write down WHY each one failed in your report.
v[v.gold_theme != v.pred_theme][["review","gold_theme","pred_theme"]].head(8)

## 7 · Insight dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17,4.8))

ct = pd.crosstab(df.pred_theme, df.pred_sentiment).reindex(columns=["negative","neutral","positive"], fill_value=0)
ct.plot(kind="barh", stacked=True, ax=axes[0], color=["#AE2F45","#B8BCCB","#0A7570"])
axes[0].set_title("Theme × sentiment"); axes[0].set_xlabel("reviews")

df.pred_urgency.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#B86E00")
axes[1].set_title("Urgency distribution (5 = act now)"); axes[1].set_xlabel("urgency")

wk = (df.assign(week=df.date.dt.to_period("W").dt.start_time)
        .groupby("week").apply(lambda g: (g.pred_sentiment=="negative").mean()*100))
wk.plot(ax=axes[2], marker="o", color="#3A49A6")
axes[2].set_title("% negative reviews by week"); axes[2].set_ylabel("%")
plt.tight_layout(); plt.show()

In [ ]:
neg = df[df.pred_sentiment=="negative"]
top = (neg.groupby("pred_theme").agg(reviews=("review","size"), avg_urgency=("pred_urgency","mean"))
          .assign(share_of_negative=lambda t: (t.reviews/t.reviews.sum()*100).round(1))
          .sort_values("reviews", ascending=False))
top

## 8 · Executive memo
An LLM drafts; **you** check every number against the table above before sending.

In [ ]:
facts = {
  "period": f"{df.date.min():%d %b %Y} to {df.date.max():%d %b %Y}",
  "total_reviews": int(len(df)),
  "pct_negative": round((df.pred_sentiment=="negative").mean()*100, 1),
  "top_negative_themes": top.head(3)[["reviews","share_of_negative","avg_urgency"]].round(2).reset_index().to_dict("records"),
  "example_quotes": neg.sort_values("pred_urgency", ascending=False)["review"].head(3).tolist(),
}
MEMO_PROMPT = ("Write a 150-word executive memo for the Head of Customer Experience. "
               "Use ONLY these facts; do not add numbers. End with 3 numbered recommendations.\n" + json.dumps(facts, default=str))

memo = None
if USE_LLM:
    try:
        throttle()
        r = client.chat.completions.create(model=LLM_MODEL, temperature=0.3,
                                           messages=[{"role":"user","content":MEMO_PROMPT}])
        memo = r.choices[0].message.content
        print(f"[Drafted by {PROVIDER} {LLM_MODEL}: check every number before sending]\n\n{memo}")
    except Exception as e:
        print("LLM memo failed, using template:", type(e).__name__)
if memo is None:
    t = facts["top_negative_themes"]
    print(f"MEMO · Customer feedback {facts['period']}\n"
          f"{facts['total_reviews']} reviews analysed; {facts['pct_negative']}% negative.\n"
          f"Top pain points: " + "; ".join(f"{x['pred_theme']} ({x['share_of_negative']}% of negatives)" for x in t) +
          "\nRecommendations: 1) Fix the top theme first  2) Review courier SLAs  3) Re-run weekly and track the trend.")

## 9 · Your turn: write-up and resume
1. **Swap in real data** (2,000 to 5,000 reviews) and rerun everything.
2. **Hand-label 200 reviews** as your gold set and report accuracy honestly.
3. **Compare two models** on accuracy, cost per 10K reviews and latency; recommend one.
4. Fill in the resume bullet **with your own numbers**:

> *Built an end-to-end GenAI pipeline that classifies product reviews by sentiment, theme and urgency using an LLM with JSON-structured outputs, reaching **[X]%** agreement with human labels on a **[N]**-review test set; discovered **[K]** complaint themes via sentence embeddings and K-means, and cut projected labelling cost to **[$Y]** per 10K reviews through token auditing and model comparison.*

**README checklist:** problem · architecture diagram · results (accuracy, cost, top themes) · 3 chart screenshots · how to run · limitations and next steps.